Finding myself frustrated with the inability to get basic logistical information on campgrounds in numerous National Forests we've been camping in this spring, I'm taking a look at how we might go about improving the public domain, open source linked open data from orginal U.S. Forest Service data. A combination of Wikidata and OpenStreetMap seem like the best base for this information content. WikiVoyage may then serve as a place to organize multiple sites within an overall context such as an area being visited or a route being explored.

Many campgrounds and designated dispersed camping areas have already been created within OSM, but their details are often lacking. Not many of these appear to have Wikidata entities, which is where we will have the greatest degree of flexibility and depth in recording useful attributes. For instance, I often need to know whether or not a campground will accommodate the length of trailer I'm pulling or if it has water service available. These details are usually present for major campgrounds that can be reserved or paid for through Recreation.gov, but it can be harder to find that information for the out of the way places we like to visit more often.

Sites like Campendium, RoadTrippers, and others I've tried all have their own proprietary data, often starting with public data but then relying on member contributions to improve details. Those sites are also heavily biased toward the most popular areas (maybe a good thing) and still lacking many crucial details I'd like to have. Furthermore, I'd rather see this information out as full public domain (or perhaps under a "NC" clause) so that anyone can take advantage of it.

I'm starting this exploration into how we might use official USFS data to figure out what's already in OSM/Wikidata for the Apache-Sitgreaves National Forest where I am now sitting in a dispersed camping area that was not on any map and is down a road that is also not on the public map sources I'm using. Nominally, the [dataset on recreation sites](https://data.fs.usda.gov/geodata/edw/edw_resources/meta/S_USA.Rec_RecreationSitesINFRA.xml) that the USFS lists as providing the basic information on their public facing web sites is probably our most comprehensive source.

Sites like [Cherry Flat Campground](https://www.openstreetmap.org/node/359263910)

In [77]:
import geopandas as gpd
import pandas as pd
import fiona
from overpy import Overpass
from shapely.geometry import Point, Polygon
import re

osm_api = Overpass()


In [2]:
gdb_fs_ranger_districts = 'data/S_USA.RangerDistrict.gdb.zip'

layers_fs_ranger_districts = fiona.listlayers(gdb_fs_ranger_districts)
df_fs_ranger_districts = gpd.read_file(gdb_fs_ranger_districts, layer=layers_fs_ranger_districts[0])

gdb_fs_rec_sites = 'data/S_USA.Rec_RecreationSitesINFRA.gdb.zip'

layers_fs_rec_sites = fiona.listlayers(gdb_fs_rec_sites)
df_fs_rec_sites = gpd.read_file(gdb_fs_rec_sites, layer=layers_fs_rec_sites[0])

In [78]:
df_fs_rec_sites['description_sections'] = df_fs_rec_sites['RECAREA_DESCRIPTION'].apply(lambda x: re.findall(r'([A-Z][A-Za-z\s]+:)', x) if x else None)

In [76]:
df_fs_rec_sites[df_fs_rec_sites['description_sections'].str.len() > 0]


,SITE_CN,ROOT_CN,PARENT_CN,REGION,SECURITY_ID,MANAGING_ORG,SITE_ID,SITE_NAME,SITE_SUBTYPE,DEVELOPMENT_STATUS,...,LMTD_USE_AWARD_METHOD,LMTD_USE_AWARD_TYPE,LATITUDE,LONGITUDE,STACKED_RECSITE_COUNT,LAST_UPDATE,GLOBALID,geometry,description,description_sections
5,1361968010602,1361968010602,None,06,0609,060903,PSTHLL,LAKE LANE,TRAILHEAD,EXISTING,...,None,None,47.464960,-123.852590,1,2022-12-14 15:49:06+00:00,{DB4B1835-B9F2-48CE-8E7D-B5D65597424D},POINT (-123.85259 47.46496),Provides access to:; Lakeshore Trail which is ...,[Provides access to:]
7,5327.005181,5327.005181,None,06,0610,061022,60250,BURNED TIMBER INTERPRETIVE,TRAILHEAD,EXISTING,...,None,None,42.514000,-123.608000,1,2022-12-14 15:49:22+00:00,{113CB978-FCBF-49B4-9380-7072E4623CE6},POINT (-123.60800 42.51400),Provides access to:; Burned Timber Interpretiv...,[Provides access to:]
13,5318.004431,5318.004431,None,05,0516,051651,51D01,NORTH FORK,PICNIC SITE,EXISTING,...,None,None,38.097544,-120.107979,1,2022-12-14 15:14:44+00:00,{41325E09-7359-4E2A-A380-A202D99EE238},POINT (-120.10798 38.09754),var $j = jQuery;$j. noConflict();$j(window). l...,"[Slide:, Speed:, Time:]"
40,5089.001481,5089.001481,None,02,0210,021007,2237,CLEAR CREEK,PICNIC SITE,EXISTING,...,None,None,39.773100,-105.768800,1,2023-10-03 16:06:48+00:00,{017CBF53-9C29-4C2E-9E68-4DC6C055A083},POINT (-105.76880 39.77310),The Clear Creek Picnic Site is located along H...,"[Facilities:, Recreation:]"
52,5026010372,5026010372,None,06,0602,060211,74006,WALT HARING SNO-PARK AND CAMPGROUND,SNOWPARK,EXISTING,...,None,None,43.228959,-121.789291,1,2023-10-03 16:07:07+00:00,{23F8741E-58D5-43EF-B122-2F0CD6AA37E3},POINT (-121.78929 43.22896),Walt Haring Sno-Park is conveniently located 5...,[Campground:]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31538,5329.005181,5329.005181,None,06,0610,061022,60252,YORK BUTTE,TRAILHEAD,EXISTING,...,None,None,42.415000,-123.809000,1,2022-12-14 15:55:39+00:00,{FA8B676E-87D8-44CC-AD5C-77910FAFA76A},POINT (-123.80900 42.41500),Provides access to:; York Butte Trail #1140; w...,[Provides access to:]
31564,150123010602,150123010602,None,05,0504,050453,53-31,MCMURRAY MEADOWS,DAY USE AREA,EXISTING,...,None,None,37.074285,-118.354247,1,2022-12-14 15:14:15+00:00,{A9D7DDCF-0E8E-468F-ACE1-F41EEB2D3D67},POINT (-118.35425 37.07429),"Aspen Group Campground is a secluded, creek si...","[Natural Features:, Recreation:, Facilities:]"
31571,9402.005181,9402.005181,None,06,0610,061026,60470,GAME LAKE,TRAILHEAD,EXISTING,...,None,None,42.437000,-124.093000,1,2022-12-14 15:49:10+00:00,{5A8FA2D9-2663-417F-B4D4-4D54A95008D2},POINT (-124.09300 42.43700),Provides access to:; Game Lake Trail #1169Faci...,[Provides access to:]
31585,5313.005181,5313.005181,None,06,0610,061022,60162,BUCKSKIN PEAK,TRAILHEAD,EXISTING,...,None,None,42.063000,-123.834000,1,2022-12-14 15:49:22+00:00,{5E86D541-CF6F-4BC6-AC57-F756748F2B30},POINT (-123.83400 42.06300),Provides access to:southern end of; Kalmiopsis...,[Provides access to:]


In [5]:
apache_sitgreaves_districts = df_fs_ranger_districts[df_fs_ranger_districts['FORESTNAME'].str.startswith('Apache-Sitgreaves')]
apache_sitgreaves_bbox = apache_sitgreaves_districts.total_bounds

In [72]:
apache_sitgreaves_rec_sites = df_fs_rec_sites.cx[apache_sitgreaves_bbox[0]:apache_sitgreaves_bbox[2], apache_sitgreaves_bbox[1]:apache_sitgreaves_bbox[3]].reset_index(drop=True)
apache_sitgreaves_rec_sites['site_name_first'] = apache_sitgreaves_rec_sites['SITE_NAME'].apply(lambda x: x.split(' ')[0].lower() if len(x.split(' ')) > 1 else x.lower())
apache_sitgreaves_rec_sites.head()

,SITE_CN,ROOT_CN,PARENT_CN,REGION,SECURITY_ID,MANAGING_ORG,SITE_ID,SITE_NAME,SITE_SUBTYPE,DEVELOPMENT_STATUS,...,LMTD_USE_AWARD_TYPE,LATITUDE,LONGITUDE,STACKED_RECSITE_COUNT,LAST_UPDATE,GLOBALID,geometry,description,description_sections,site_name_first
0,5220.001931,5220.001931,None,03,0301,030102,46653,GENTRY CG,CAMPGROUND,EXISTING,...,None,34.301531,-110.712222,1,2023-04-24 14:59:55+00:00,{BF525784-7531-4B74-8287-D21C8F8D782C},POINT (-110.71222 34.30153),The campground is set among the ponderosa pine...,[],gentry
1,34875010602,34875010602,None,03,0301,030102,46671,WILDCAT ROAD OHV ACCESS,DAY USE AREA,EXISTING,...,None,34.351060,-110.811269,1,2022-12-14 15:13:26+00:00,{95958293-58FA-4A27-976D-AE6B293CBEC6},POINT (-110.81127 34.35106),Wildcat Road OHV Access Day Use Area,[],wildcat
2,5461.001931,5461.001931,None,03,0301,030103,45273,HORSE CANYON TRAILHEAD (#36),TRAILHEAD,EXISTING,...,None,33.326478,-109.189056,1,2022-12-14 15:13:30+00:00,{9C7E42D9-954F-417E-AD1D-EBF4687CFBCA},POINT (-109.18906 33.32648),Horse Canyon Trailhead (#36) Trailhead,[],horse
3,5228.001931,5228.001931,None,03,0301,030106,46902,GRAYLING CG,CAMPGROUND,EXISTING,...,None,33.872919,-109.412928,1,2023-10-03 16:06:55+00:00,{03AF6865-C4E8-4C1D-9B0D-19F15882603E},POINT (-109.41293 33.87292),"Situated in the mixed conifer and aspen, Grayl...","[Natural Features:, Recreation:, Facilities:]",grayling
4,5232.001931,5232.001931,None,03,0301,030106,46906,GOVERNMENT SPRINGS,TRAILHEAD,EXISTING,...,None,33.993989,-109.464717,1,2023-10-03 16:06:55+00:00,{61A2B351-28D8-4592-8CE4-738E92B1E9E9},POINT (-109.46472 33.99399),None,None,government


In [30]:
bbox = apache_sitgreaves_bbox.tolist()

query_osm_cg = f"""
[out:json];
(
    node["tourism"~"camp_site"]({bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]});
    way["tourism"~"camp_site"]({bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]});
);
out;
"""

response = osm_api.query(query_osm_cg)

In [48]:
osm_nodes = []

for node in response.nodes:
    item = {
        'id': node.id,
        'lat': node.lat,
        'lon': node.lon,
        'geometry': Point(float(node.lon), float(node.lat)),
    }
    item.update(node.tags)
    osm_nodes.append(item)

osm_ways = []

for way in response.ways:
    item = {
        'id': way.id,
    }
    item.update(way.tags)
    osm_ways.append(item)

In [49]:
gdf_osm_data = gpd.GeoDataFrame(pd.DataFrame(osm_nodes), crs='EPSG:4326', geometry='geometry')
df_osm_ways = pd.DataFrame(osm_ways)

In [57]:
osm_campsites_apache_sitegreaves = pd.concat(
    [
        gdf_osm_data[['id', 'name', 'operator','source']],
        df_osm_ways[['id', 'name', 'operator','source']]
    ]
)
osm_campsites_apache_sitegreaves['site_name_first'] = osm_campsites_apache_sitegreaves['name'].apply(lambda x: x.split(' ')[0].lower() if isinstance(x, str) else None)

In [62]:
pd.merge(
    left=apache_sitgreaves_rec_sites,
    right=osm_campsites_apache_sitegreaves,
    on='site_name_first',
    how='inner',
)[['SITE_NAME','SITE_SUBTYPE','name','operator','source']]

,SITE_NAME,SITE_SUBTYPE,name,operator,source
0,GENTRY CG,CAMPGROUND,Gentry Campground,USFS,NaN
1,HORSE CANYON TRAILHEAD (#36),TRAILHEAD,Horse Springs Campground & Group Ramada,NaN,NaN
2,GRAYLING CG,CAMPGROUND,Grayling Campground,USFS,www.fs.usda.gov
3,LAKESIDE,CAMPGROUND,Lakeside Campground,USFS,www.fs.usda.gov
4,CANYON POINT AMPHITHEATHER,INTERPRETIVE SITE,Canyon Point Campground,ExplorUS,NaN
...,...,...,...,...,...
219,WOODS CANYON AMPHITHEATER,INTERPRETIVE SITE,Woods Canyon Group Campground,USFS,www.fs.usda.gov
220,BLUE CROSSING CG,CAMPGROUND,Blue Crossing Campground,USFS,local knowledge
221,BROWN CREEK,CAMPING AREA,Brown Creek Campground,USFS,www.fs.usda.gov
222,LOS BURROS #1,TRAILHEAD,Los Burros Campground,USFS,www.fs.usda.gov


In [73]:
apache_sitgreaves_rec_sites[apache_sitgreaves_rec_sites['SITE_NAME'] == 'GRAYLING CG'].iloc[0]['description']

"Situated in the mixed conifer and aspen, Grayling Campground offers perhaps the most secluded sites in the Big Lake complex. It sits on the shoreline of the lake in a mixed conifer and aspen forest and affords visitors many recreational opportunities in a quiet, scenic setting. Popular activities include trout fishing and hiking. Natural Features:Grayling sits in the White Mountains at an elevation of 9,000 feet. It is a very scenic area, forested with ponderosa pine, Douglas fir, spruce and aspen trees, which provide ample shade. Summer temperatures are pleasant and cool, a nice respite from the heat of lower deserts. Visitors should come prepared for cool nighttime temperatures and occasional summer rain storms. Recreation:The campground is adjacent to 450-acre Big Lake, one of Arizona 's top fishing lakes. The lake is known for its trout fishing, with rainbow, cutthroat, brook and Apache being the most sought-after species. The recreation area is also popular for hiking, mountain b